In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', None)

In [4]:
ROOT_DIR = "/content/drive/MyDrive/Colab Notebooks/DA/lgaimers/data"
RANDOM_STATE = 110

##데이터 로드

In [5]:
# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))

##결측값 대체

In [6]:
# Fill missing values
train_data.fillna(0, inplace=True)

##데이터 타입 변환

In [13]:
# Ensure consistent data types
for col in train_data.columns:
    if train_data[col].dtype == 'object':
        try:
            train_data[col] = pd.to_numeric(train_data[col])
        except ValueError:
            pass

## 단일 고유값을 가지는 칼럼 삭제

In [14]:
# Drop columns with a single unique value
one_val_duplicated = [col for col in train_data.columns if train_data[col].nunique() == 1]
train_data.drop(columns=one_val_duplicated, inplace=True)

## 모든 행이 고유값을 가지는 칼럼 삭제 (예: ID와 같은 칼럼)

In [15]:
# Drop columns with unique values for every row (e.g., IDs)
num_rows = len(train_data)
unique_every_row = [col for col in train_data.columns if train_data[col].nunique() == num_rows]
train_data.drop(columns=unique_every_row, inplace=True)

## 문자열로 된 범주형 변수들을 숫자로 변환하기 위해 LabelEncoder 사용

In [16]:
# Convert categorical features to numeric using LabelEncoder
label_encoders = {}
for col in train_data.columns:
    if train_data[col].dtype == 'object':
        le = LabelEncoder()
        train_data[col] = le.fit_transform(train_data[col].astype(str))
        label_encoders[col] = le

## 피처와 타겟 변수 분리

In [17]:
# Split data into features and target
X = train_data.drop(columns=['target'])
y = train_data['target']


## 타겟 변수에 대한 레이블 인코딩

In [18]:
# Label encoding for the target variable
le = LabelEncoder()
y = le.fit_transform(y)

## SMOTE를 사용하여 클래스 불균형 처리

In [19]:
# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = smote.fit_resample(X, y)

In [22]:
# Feature selection using mutual information
selector = SelectKBest(mutual_info_classif, k=20)  # Select top 20 features
X_res_selected = selector.fit_transform(X_res, y_res)

In [23]:
# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_res, y_res, test_size=0.3, stratify=y_res, random_state=RANDOM_STATE)

## 분류 모델 정의

In [24]:
# Define the models
pre_cal_xgbc = XGBClassifier(n_estimators=300, learning_rate=0.01, max_depth=5, min_child_weight=4,
                             subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
                             n_jobs=-1, gamma=0.3, eval_metric='logloss')

pre_cal_rf = RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=4,
                                    min_samples_split=10, n_jobs=-1, random_state=RANDOM_STATE)

pre_cal_svc = SVC(probability=True, random_state=RANDOM_STATE, class_weight='balanced', kernel='rbf')

## 모델 보정

In [25]:
# Calibrate classifiers
xgbc = CalibratedClassifierCV(pre_cal_xgbc, method='isotonic', cv=5)
rfc = CalibratedClassifierCV(pre_cal_rf, method='isotonic', cv=5)
svc = CalibratedClassifierCV(pre_cal_svc, method='sigmoid', cv=5)

## 모델 학습

In [ ]:
# Train the models
xgbc.fit(X_train, y_train)
rfc.fit(X_train, y_train)
svc.fit(X_train, y_train)

## 앙상블 투표 기반 모델 정의

In [ ]:
# Voting classifier
model = VotingClassifier(estimators=[('rf', rfc), ('xgb', xgbc), ('svc', svc)],
                         voting='soft', weights=[2, 3, 1])


## 앙상블 모델 학습

In [ ]:
# Fit the ensemble model
model.fit(X_train, y_train)

## 검증 데이터로 예측

In [ ]:
# Validate the model
y_pred = model.predict(X_val)
f1 = f1_score(y_val, y_pred)
print(f"Validation F1 Score: {f1:.4f}")

## 테스트 데이터 로드

In [ ]:
# Load test data
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))
test_data.fillna(0, inplace=True)

In [ ]:
# Ensure consistent data types in test data
for col in test_data.columns:
    if test_data[col].dtype == 'object':
        try:
            test_data[col] = pd.to_numeric(test_data[col])
        except ValueError:
            pass

In [ ]:
# Apply the same label encoding for categorical features in test data
for col, le in label_encoders.items():
    if col in test_data.columns:
        test_data[col] = le.transform(test_data[col].astype(str))

In [ ]:
# Apply same feature selection on test data
test_X = selector.transform(test_data)

## 테스트 데이터로 예측 수행

In [ ]:
# Predict on test data
test_pred = model.predict(test_X)

## 제출

In [ ]:
# Prepare submission
submission = pd.read_csv(os.path.join("/content/drive/MyDrive/Colab Notebooks/DA/lgaimers/submission.csv")
submission['target'] = le.inverse_transform(test_pred)
submission.to_csv("/content/drive/MyDrive/Colab Notebooks/DA/lgaimers/submission.csv", index=False)